# 03/08 — Rexach BA4_MIC-7 (ITM2B⁺ "resilience" microglia) in the BRICHOS ST data

**Question (from the Rexach *Cell* 2024 paper + the email).**
Rexach et al. describe **BA4_MIC-7**, an AD-specific microglial state in
**motor cortex — a region relatively *spared* in AD**. It is enriched for
amyloid processing, chaperone-mediated autophagy and oxidative-stress
buffering, its up-genes carry AD GWAS risk, and its **top up-marker is
ITM2B (Bri2) — the precursor of the BRICHOS domain we dosed into these
APP-NLGF mice**. The authors read it as a *protective / resilience* state
(Fig 2C, 2E–G; Fig 6C).

**Hypothesis to probe here.** Does BRICHOS treatment shift cortical
microglia toward a BA4_MIC-7-like program (↑ITM2B, ↑amyloid-processing /
CMA / oxidative buffering), region-by-region, in our Visium data?

**Honest caveats baked into this notebook.**
1. *Cross-species + cross-modality.* Human snRNA-seq microglia state →
   mouse Visium 55 µm spots (bulk-per-spot; microglia are a minority of
   each spot). Signal is diluted and confounded by **microglial density**
   — we control for it explicitly.
2. *Ambient contamination.* In Rexach's strict DE list (padj<0.05, n=17
   up-genes) several top hits are oligo/neuronal (**PLP1, GRID2, CNTN2,
   DPYSL2**) — almost certainly ambient RNA in the nuclei. We carry a
   contamination-pruned **core** signature alongside the full set.
3. *Overlap with generic activation.* Part of BA4_MIC-7 (Apoe, Bin1, C3,
   Cd81) overlaps the DAM/PIG disease-associated program that is already
   up in our PBS mice. The *separable, interesting* part is the
   resilience-specific subset — we test it both ways.
4. *Direction is the open question.* BA4_MIC-7 is empirically UP in AD vs
   control in Rexach, but interpreted as protective. So whether BRICHOS
   should push it **up** (more resilience) or **down** (less inflammation)
   is genuinely unknown — we report the BRI-vs-PBS direction, we do not
   assume it.

Outputs: per-spot scores added to `.obs`, a region×signature rescue table,
a mixed-model table, responder maps, and figures under
`results/figures/manuscript/` + tables under `results/tables/attenuation/`.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

# Rexach 2024 Table S5 raw DE table (BA4_MIC-7 AD vs all other microglia)
REXACH_DE = ROOT / 'data' / 'external' / 'rexach2024_BA4_MIC7_AD_DE.tsv'

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'
REGION_KEY    = 're_annotation_regions'
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('rexach DE   :', REXACH_DE, '(exists:', REXACH_DE.exists(), ')')

## 1 · Build the BA4_MIC-7 signatures (human → mouse)

We take Rexach's authoritative DE list (Table S5, sheet *"BA4_MIC-7 AD"*),
keep the significantly **up** genes (padj<0.05), and map human→mouse with
`utils.orthologs`. The curated signatures already live in
`utils.gene_sets` (so `03/07-broader-signatures` picks them up
automatically); here we also re-derive the set straight from the raw table
as a provenance check and print the full mapping.

In [ ]:
from utils import gene_sets as gs
from utils.orthologs import to_mouse, mapping_table

# --- re-derive from the raw Table S5 to verify provenance --------------
de = pd.read_csv(REXACH_DE, sep='\t')
de.columns = ['gene', 'lfc', 'pvalue', 'padj']
up_hs = de.query('lfc > 0 and padj < 0.05').sort_values('lfc', ascending=False)
dn_hs = de.query('lfc < 0 and padj < 0.05')
print(f'Rexach BA4_MIC-7: {len(up_hs)} up / {len(dn_hs)} down (padj<0.05) '
      f'of {len(de)} tested')

print('\nhuman -> mouse mapping (up-genes):')
mt = mapping_table(up_hs['gene'])
print(pd.DataFrame([(h, '/'.join(m), src) for h, m, src in mt],
                   columns=['human', 'mouse', 'source']).to_string(index=False))

# sanity: the registry set equals the re-derived mapping
assert set(to_mouse(up_hs['gene'])) == set(gs.BA4_MIC7_UP_REXACH2024), \
    'registry set drifted from the raw table — regenerate gene_sets'
print('\n✓ utils.gene_sets.BA4_MIC7_UP_REXACH2024 matches the raw table')

In [ ]:
# Signatures we will score, with explicit provenance.
#   full     : all padj<0.05 up genes (incl. likely ambient PLP1/GRID2/...)
#   core     : contamination-pruned microglial subset
#   amyloid / CMA / oxidative : Fig-2E functional sub-programs (mechanism)
SIGNATURES = {
    'BA4_MIC7_full':      gs.BA4_MIC7_UP_REXACH2024,
    'BA4_MIC7_core':      gs.BA4_MIC7_RESILIENCE_CORE,
    'BA4_MIC7_amyloid':   gs.BA4_MIC7_AMYLOID_PROCESSING,
    'BA4_MIC7_CMA':       gs.BA4_MIC7_CMA,
    'BA4_MIC7_oxidative': gs.BA4_MIC7_OXIDATIVE,
}

# Pan-microglia CONTENT proxy — state-independent myeloid genes, used to
# control for how much microglia each spot contains (the key confound when
# porting a single-cell microglial state onto bulk-ish Visium spots).
MICROGLIA_CONTENT = ['Csf1r', 'C1qa', 'C1qb', 'C1qc', 'Fcer1g', 'Ctss',
                     'Tyrobp', 'Aif1', 'Hexb']

for name, genes in SIGNATURES.items():
    print(f'  {name:18} {len(genes)} genes')
print('signatures defined:', list(SIGNATURES))

## 2 · Load data & score per spot (with microglia-density control)

All scoring uses `sc.tl.score_genes` (the project convention). We add the
BA4_MIC-7 scores, a microglia-content score, the DAM/PIG references for
later comparison, and pull out **Itm2b** expression directly.

In [ ]:
adata = sc.read_h5ad(H5AD)
print(adata)

def cov(genes):
    present = gs.filter_to_var(genes, adata.var_names)
    return present

# score BA4_MIC-7 signatures (skip if <3 genes present)
for name, genes in SIGNATURES.items():
    present = cov(genes)
    if len(present) < 3:
        print(f'⚠️  {name}: only {len(present)}/{len(genes)} genes in data — skipped')
        continue
    sc.tl.score_genes(adata, present, score_name=f'{name}_score', use_raw=False)
    print(f'✓ {name}: scored {len(present)}/{len(genes)}  {present}')

# microglia content + reference disease programs
for name, genes in [('microglia_content', MICROGLIA_CONTENT),
                    ('PIG', gs.PIG_CHEN2020),
                    ('DAM', gs.DAM_KEREN_SHAUL2017),
                    ('microglia_homeo', gs.MICROGLIA_HOMEOSTATIC)]:
    present = cov(genes)
    sc.tl.score_genes(adata, present, score_name=f'{name}_score', use_raw=False)
    print(f'✓ {name}: {len(present)}/{len(genes)}')

# Itm2b directly (the headline gene)
ITM2B_IN = 'Itm2b' in adata.var_names
print('\nItm2b in panel:', ITM2B_IN)
if ITM2B_IN:
    adata.obs['Itm2b_expr'] = np.asarray(
        adata[:, 'Itm2b'].X.todense()).ravel() if hasattr(
        adata[:, 'Itm2b'].X, 'todense') else np.asarray(adata[:, 'Itm2b'].X).ravel()

## 3 · Itm2b — the headline gene

If there is one thing to look at, it is *Itm2b* itself. We ask: is it
induced by AD (PBS vs WT)? Is it moved by BRICHOS (BRI vs PBS)? And where,
spatially?

In [ ]:
from utils.attenuation import make_pseudobulk, pseudobulk_lfc

counts, meta = make_pseudobulk(adata, SAMPLE_KEY, REGION_KEY,
                               TREATMENT_KEY, min_spots=20, layer=COUNT_LAYER)
print('pseudobulk:', counts.shape, '| groups per treatment:')
print(meta.groupby('treatment')['sample'].nunique())

# Itm2b pseudobulk LFC per region, both contrasts (descriptive; WT n=1)
itm_rows = []
for region in sorted(meta['region'].unique()):
    rec = {'region': region}
    for tag, (a, b) in {'AD(PBSvWT)': ('PBS', 'WT'),
                        'BRIvPBS':   ('BRICHOS', 'PBS')}.items():
        try:
            res = pseudobulk_lfc(counts, meta, a, b, region=region,
                                 min_n=1).to_frame()
            rec[f'{tag}_lfc'] = res.loc['Itm2b', 'lfc'] if 'Itm2b' in res.index else np.nan
            rec[f'{tag}_padj'] = res.loc['Itm2b', 'padj'] if 'Itm2b' in res.index else np.nan
        except Exception as e:
            rec[f'{tag}_lfc'] = np.nan; rec[f'{tag}_padj'] = np.nan
    itm_rows.append(rec)
itm_df = pd.DataFrame(itm_rows).set_index('region')
itm_df.to_csv(TBL / 'itm2b_regional_lfc.tsv', sep='\t')
itm_df.round(3)

In [ ]:
# Itm2b expression by treatment x region (dotplot proxy via groupby means)
if ITM2B_IN:
    piv = (adata.obs.assign(Itm2b=adata.obs['Itm2b_expr'])
           .groupby([REGION_KEY, TREATMENT_KEY], observed=True)['Itm2b']
           .mean().unstack())
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(piv.values, aspect='auto', cmap='magma')
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
    ax.set_title('Mean Itm2b (log-norm) by region × treatment')
    fig.colorbar(im, ax=ax, shrink=0.6)
    fig.tight_layout()
    fig.savefig(FIG / 'itm2b_region_treatment.svg', bbox_inches='tight')
    fig.savefig(FIG / 'itm2b_region_treatment.png', dpi=200, bbox_inches='tight')
    plt.show()
    display(piv.round(3))

In [ ]:
# Spatial Itm2b map, per treatment composite (mirrors 03/04 plotting)
if ITM2B_IN:
    vmax = float(np.nanpercentile(adata.obs['Itm2b_expr'], 99))
    for treat in ['WT', 'PBS', 'BRICHOS']:
        sub = adata[adata.obs[TREATMENT_KEY] == treat]
        if sub.n_obs == 0:
            continue
        libs = sub.obs[SAMPLE_KEY].unique()
        fig, axes = plt.subplots(1, len(libs), figsize=(3 * len(libs), 3.2),
                                 squeeze=False)
        for ax, lib in zip(axes.flat, libs):
            sl = sub[sub.obs[SAMPLE_KEY] == lib]
            sp = adata.uns['spatial'][lib]
            sf = sp['scalefactors']['tissue_hires_scalef']
            ax.imshow(sp['images']['hires'], origin='upper')
            xy = sl.obsm['spatial'] * sf
            ax.scatter(xy[:, 0], xy[:, 1], c=sl.obs['Itm2b_expr'].values,
                       s=0.6, cmap='magma', vmin=0, vmax=vmax, edgecolors='none')
            ax.set_title(f'{lib} [{treat}]', fontsize=8, loc='left')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
        fig.suptitle(f'{treat}: Itm2b expression', fontsize=10)
        fig.tight_layout()
        fig.savefig(FIG / f'itm2b_spatial_{treat}.png', dpi=200, bbox_inches='tight')
        plt.show()

## 4 · Per-region rescue of the BA4_MIC-7 program

Reuse the attenuation framework: per region compute the **disease** LFC
(PBS vs WT) and the **rescue** LFC (BRI vs PBS) over the BA4_MIC-7 genes,
and test whether BRICHOS moves the program. Because BA4_MIC-7 is read as
protective, we report the *signed* BRI-vs-PBS median LFC (positive = BRICHOS
*raises* the program) rather than forcing the disease-down prior.

In [ ]:
from utils.attenuation import signed_rescue

sig_genes = {k: cov(v) for k, v in SIGNATURES.items()}
rows = []
for region in sorted(meta['region'].unique()):
    n_bri = (meta.query('region == @region')['treatment'] == 'BRICHOS').sum()
    n_pbs = (meta.query('region == @region')['treatment'] == 'PBS').sum()
    if n_bri < 2 or n_pbs < 2:
        continue
    rescue = pseudobulk_lfc(counts, meta, 'BRICHOS', 'PBS',
                            region=region, min_n=2).lfc
    for sname, genes in sig_genes.items():
        g = [x for x in genes if x in rescue.index]
        if len(g) < 3:
            continue
        med = float(rescue.loc[g].median())
        # is the BRI-vs-PBS shift over these genes != 0 ? (Wilcoxon)
        from scipy.stats import wilcoxon
        try:
            p = wilcoxon(rescue.loc[g]).pvalue
        except ValueError:
            p = np.nan
        rows.append(dict(region=region, signature=sname, n_genes=len(g),
                         median_BRIvPBS_lfc=med, wilcoxon_p=p,
                         n_bri=n_bri, n_pbs=n_pbs))
rescue_df = pd.DataFrame(rows)
rescue_df.to_csv(TBL / 'ba4_mic7_regional_rescue.tsv', sep='\t', index=False)
rescue_df.round(3)

In [ ]:
# Heatmap: region × signature median BRI-vs-PBS LFC (red = BRICHOS raises it)
if len(rescue_df):
    H = rescue_df.pivot(index='region', columns='signature',
                        values='median_BRIvPBS_lfc')
    vlim = float(np.nanmax(np.abs(H.values)))
    fig, ax = plt.subplots(figsize=(1.1 * H.shape[1] + 2, 0.5 * H.shape[0] + 2))
    im = ax.imshow(H.values, cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto')
    ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns, rotation=45, ha='right')
    ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index, fontsize=8)
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            v = H.values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7)
    ax.set_title('BA4_MIC-7 programs: median BRI–PBS LFC by region\n(red = BRICHOS raises)')
    fig.colorbar(im, ax=ax, shrink=0.6, label='log2FC (BRI – PBS)')
    fig.tight_layout()
    fig.savefig(FIG / 'ba4_mic7_rescue_heatmap.svg', bbox_inches='tight')
    fig.savefig(FIG / 'ba4_mic7_rescue_heatmap.png', dpi=200, bbox_inches='tight')
    plt.show()

## 5 · Mixed-effects model — controlling for microglia density

The per-spot score can rise simply because a spot has more microglia. We
fit `score ~ treatment * region + microglia_content + (1 | mouse)` so the
treatment effect is *adjusted* for microglial content (and region). Mirrors
`03/03-signature-mixed-model`.

In [ ]:
import statsmodels.formula.api as smf

def fit_mixed(score_col):
    df = adata.obs[[TREATMENT_KEY, REGION_KEY, SAMPLE_KEY,
                    'microglia_content_score', score_col]].dropna().copy()
    df = df.rename(columns={TREATMENT_KEY: 'treatment', REGION_KEY: 'region',
                            SAMPLE_KEY: 'mouse', score_col: 'score',
                            'microglia_content_score': 'mg'})
    # PBS as reference so coefficients read as "vs PBS"
    df['treatment'] = pd.Categorical(df['treatment'],
                                     categories=['PBS', 'WT', 'BRICHOS'])
    md_ = smf.mixedlm('score ~ C(treatment) * C(region) + mg', df,
                      groups=df['mouse'])
    return md_.fit(method='lbfgs', maxiter=200)

mm_rows = []
for score_col in [f'{k}_score' for k in SIGNATURES if f'{k}_score' in adata.obs]:
    try:
        res = fit_mixed(score_col)
        for term in res.params.index:
            if 'BRICHOS' in term or term == 'mg':
                mm_rows.append(dict(signature=score_col, term=term,
                                    coef=res.params[term], p=res.pvalues[term]))
    except Exception as e:
        print(f'{score_col}: model failed — {e}')
mm_df = pd.DataFrame(mm_rows)
mm_df.to_csv(TBL / 'ba4_mic7_mixedmodel.tsv', sep='\t', index=False)
# show the main-effect of BRICHOS and the microglia covariate per signature
mm_df[mm_df['term'].isin(['C(treatment)[T.BRICHOS]', 'mg'])].round(4)

## 6 · Responder map for the BA4_MIC-7 core program

Same device as `03/04` but on the BA4_MIC-7 core score: per spot, z-shift
relative to the regional **PBS** distribution. Here a **positive** z under
BRICHOS = a spot pushed *toward* the (putatively protective) BA4_MIC-7
state. We map it and quantify the fraction per region (BRI vs PBS).

In [ ]:
from utils.attenuation import responder_zshift
from scipy.stats import mannwhitneyu

SCORE = 'BA4_MIC7_core_score'
if SCORE in adata.obs:
    adata.obs['MIC7_zshift_vsPBS'] = responder_zshift(
        adata, score_key=SCORE, treatment_key=TREATMENT_KEY,
        region_key=REGION_KEY, sample_key=SAMPLE_KEY, ref='PBS')
    # "induced" spot = pushed up toward MIC-7 (z > +1 vs regional PBS)
    adata.obs['MIC7_induced'] = adata.obs['MIC7_zshift_vsPBS'] > 1.0

    rows = []
    for region in adata.obs[REGION_KEY].dropna().unique():
        bm = (adata.obs[adata.obs[REGION_KEY] == region]
              .groupby(SAMPLE_KEY, observed=True)
              .agg(treatment=(TREATMENT_KEY, 'first'),
                   frac=('MIC7_induced', 'mean')))
        pbs = bm.loc[bm['treatment'] == 'PBS', 'frac'].values
        bri = bm.loc[bm['treatment'] == 'BRICHOS', 'frac'].values
        if len(pbs) >= 2 and len(bri) >= 2:
            _, p = mannwhitneyu(bri, pbs, alternative='greater')
        else:
            p = np.nan
        rows.append(dict(region=region, mean_pbs=np.mean(pbs) if len(pbs) else np.nan,
                         mean_bri=np.mean(bri) if len(bri) else np.nan, mwu_p=p))
    mic7_resp = pd.DataFrame(rows).set_index('region').sort_values('mwu_p')
    mic7_resp.to_csv(TBL / 'ba4_mic7_induced_fraction_by_region.tsv', sep='\t')
    display(mic7_resp.round(3))

In [ ]:
# Spatial responder map (per-treatment composite)
if SCORE in adata.obs:
    for treat in ['WT', 'PBS', 'BRICHOS']:
        sub = adata[adata.obs[TREATMENT_KEY] == treat]
        if sub.n_obs == 0:
            continue
        libs = sub.obs[SAMPLE_KEY].unique()
        fig, axes = plt.subplots(1, len(libs), figsize=(3 * len(libs), 3.2),
                                 squeeze=False)
        for ax, lib in zip(axes.flat, libs):
            sl = sub[sub.obs[SAMPLE_KEY] == lib]
            sp = adata.uns['spatial'][lib]
            sf = sp['scalefactors']['tissue_hires_scalef']
            ax.imshow(sp['images']['hires'], origin='upper')
            xy = sl.obsm['spatial'] * sf
            c = sl.obs['MIC7_zshift_vsPBS'].clip(-3, 3).values
            ax.scatter(xy[:, 0], xy[:, 1], c=c, s=0.6, cmap='coolwarm',
                       vmin=-3, vmax=3, edgecolors='none')
            ax.set_title(f'{lib} [{treat}]', fontsize=8, loc='left')
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(False)
        fig.suptitle(f'{treat}: BA4_MIC-7 core z-shift vs regional PBS '
                     f'(red = toward MIC-7)', fontsize=10)
        fig.tight_layout()
        fig.savefig(FIG / f'mic7_zshift_{treat}.png', dpi=200, bbox_inches='tight')
        plt.show()

## 7 · Is BA4_MIC-7 separable from generic DAM/PIG activation?

A big chunk of the signature overlaps the disease-associated microglia
program already up in our PBS mice. We quantify (a) gene-level overlap and
(b) per-spot score correlation. If the BA4_MIC-7 score is just a proxy for
PIG/DAM, any "rescue" is not specific to the resilience program.

In [ ]:
def jaccard(a, b):
    A, B = set(a), set(b)
    return len(A & B) / len(A | B) if (A | B) else np.nan

print('Gene overlap (Jaccard) of BA4_MIC7_full with:')
for ref, genes in [('PIG', gs.PIG_CHEN2020), ('DAM', gs.DAM_KEREN_SHAUL2017),
                   ('homeostatic', gs.MICROGLIA_HOMEOSTATIC)]:
    j = jaccard(gs.BA4_MIC7_UP_REXACH2024, genes)
    shared = sorted(set(gs.BA4_MIC7_UP_REXACH2024) & set(genes))
    print(f'  {ref:12} J={j:.3f}  shared={shared}')

# per-spot score correlations
cols = [c for c in ['BA4_MIC7_full_score', 'BA4_MIC7_core_score',
                    'PIG_score', 'DAM_score', 'microglia_content_score']
        if c in adata.obs]
corr = adata.obs[cols].corr(method='spearman')
display(corr.round(2))

# partial: does BA4_MIC7_core still separate treatments after removing
# microglia_content and PIG? (residualize, then group means)
from numpy.linalg import lstsq
def residualize(y, X):
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, *_ = lstsq(X1, y, rcond=None)
    return y - X1 @ beta

if 'BA4_MIC7_core_score' in adata.obs:
    ok = adata.obs[['BA4_MIC7_core_score', 'microglia_content_score',
                    'PIG_score', TREATMENT_KEY]].dropna()
    resid = residualize(ok['BA4_MIC7_core_score'].values,
                        ok[['microglia_content_score', 'PIG_score']].values)
    ok = ok.assign(resid=resid)
    print('\nBA4_MIC7_core residual (after removing microglia_content + PIG), '
          'mean by treatment:')
    display(ok.groupby(TREATMENT_KEY, observed=True)['resid'].agg(['mean', 'count']).round(4))

## 8 · Read-out / interpretation scaffold

Fill in once run against the mounted data. The decision tree:

- **Itm2b (§3).** Induced in PBS-vs-WT? Moved by BRICHOS? Where? This is the
  most direct, least model-dependent result and the one most defensible to
  show given the ITM2B-centric story in Rexach.
- **Rescue heatmap (§4) + mixed model (§5).** Does BRICHOS raise the
  BA4_MIC-7 *core* / *amyloid* / *CMA* / *oxidative* programs specifically
  in cortex (supra/infragranular layers — the closest analogue to spared
  motor cortex)? The `mg` covariate must be non-trivial, else the score is
  just microglial density.
- **Responder map (§6).** Spatial coherence of MIC-7-induced spots.
- **Specificity (§7).** If BA4_MIC7_core correlates ~1 with PIG/DAM and the
  residual shows no treatment separation, report it honestly as
  *not separable from generic activation* in this bulk-per-spot data.

**Bottom line to write up:** this is a *hypothesis-generating* port of a
human snRNA-seq state onto mouse Visium, not a validation. The strongest
claim the data can support is "Itm2b and the BA4_MIC-7 resilience program
are [in/de]-creased by BRICHOS in [regions], independent of microglial
density" — or the honest null.